In [11]:

import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

# Gemini model configuration
llm_config = {
    "model": "gemini-3.5-flash",
    "api_key": GEMINI_API_KEY,
}

print("Gemini API key loaded successfully!")


Gemini API key loaded successfully!


In [13]:

import os
from dotenv import load_dotenv

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in your .env file.")

model_client = OpenAIChatCompletionClient(
    model="gemini-3.5-flash",
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": True,
    },
)

onboarding_personal_information_agent = AssistantAgent(
    name="onboarding_personal_information_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful customer onboarding agent. "
        "Gather the customer's name and location only. "
        "Do not ask for other information. "
        "When both details have been gathered, return TERMINATE."
    ),
)

onboarding_topic_preference_agent = AssistantAgent(
    name="onboarding_topic_preference_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful customer onboarding agent. "
        "Gather the customer's preferences on news topics only. "
        "Do not ask for other information. "
        "When preferences have been gathered, return TERMINATE."
    ),
)

customer_engagement_agent = AssistantAgent(
    name="customer_engagement_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful customer service agent. "
        "Use the user's personal information and topic preferences "
        "to provide engaging fun facts, jokes, or interesting stories. "
        "Keep the interaction fun. Return TERMINATE when done."
    ),
)

customer_proxy_agent = UserProxyAgent(
    name="customer_proxy_agent",
    input_func=input,
)

print("Gemini onboarding agents created successfully!")



Gemini onboarding agents created successfully!


In [14]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

In [15]:
import asyncio
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

team = RoundRobinGroupChat(
    participants=[
        onboarding_personal_information_agent,
        onboarding_topic_preference_agent,
        customer_engagement_agent
    ],
    termination_condition=MaxMessageTermination(6)
)

result = await team.run(
    task="Start the customer onboarding conversation."
)

for message in result.messages:
    if hasattr(message, "content"):
        print(message.content)

Start the customer onboarding conversation.
Hello! Welcome. To get started with your onboarding, could you please tell me your full name?
Hello and welcome! I am here to help you set up your account preferences. 

To personalize your feed, could you please let me know what news topics you are interested in following (for example: Technology, Business, World News, Sports, Entertainment, Science)?
Hello and a super warm welcome! 🎉 I am thrilled to help you get set up today! 

To get your personalized experience rolling, could you please share:
1. **Your full name**
2. **Your favorite topics or interests** (for example: Technology, Science, Sports, Business, Entertainment, World News, or space trivia!)

Once I have those, I'll tune your feed and share some awesome facts and jokes tailored just for you!
Welcome! I would be happy to help you get onboarded. 

To get started, could you please provide your full name and your current location (city/state or country)?
Hello! I am focused solely 

In [17]:
chat_results = [result]

summary = []

for chat_result in chat_results:
    summary_text = chat_result.messages[-1].content
    print(summary_text)
    summary.append(summary_text)
    print("\n")

Hello! I am focused solely on setting up your news feed today. 

Could you please share which news topics you are most interested in? For example: Technology, Business, World News, Sports, Science, or Entertainment.




In [18]:
summary[0]

'Hello! I am focused solely on setting up your news feed today. \n\nCould you please share which news topics you are most interested in? For example: Technology, Business, World News, Sports, Science, or Entertainment.'

In [20]:
import pandas as pd

df = pd.DataFrame({
    "Summary": summary
})

df

,Summary
0,Hello! I am focused solely on setting up your ...


In [21]:
data_dict = {
    "message": summary[0]
}

df = pd.DataFrame([data_dict])
df

,message
0,Hello! I am focused solely on setting up your ...


In [23]:
df["TopicPreference"]=summary[0]

In [24]:
df

,message,TopicPreference
0,Hello! I am focused solely on setting up your ...,Hello! I am focused solely on setting up your ...


In [16]:
df["Enagement"]=summary[2]

In [17]:
df

,name,location,TopicPreference,Enagement
0,Ramisha,Theni,Ramisha from Theni is interested in reading ab...,"According to the Law of Attraction, positive o..."


In [20]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'websiteleadchatgptapi-a46f50ae1507.json', scope
    )


    client = gspread.authorize(creds)

    sheet_id = '1w-19wp0W2nRV5Pde5gim6kKlCoPTjv6mT3A8lAdNrcM'

    spreadsheet = client.open_by_key(sheet_id)

    worksheet = spreadsheet.get_worksheet(0)

    for _, row in df.iterrows():
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [21]:
add_leads_to_google_sheet(df)

Data added successfully!


'added'

In [22]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0.000115, 'gpt-3.5-turbo-0125': {'cost': 0.000115, 'prompt_tokens': 140, 'completion_tokens': 30, 'total_tokens': 170}}, 'usage_excluding_cached_inference': {'total_cost': 5.75e-05, 'gpt-3.5-turbo-0125': {'cost': 5.75e-05, 'prompt_tokens': 70, 'completion_tokens': 15, 'total_tokens': 85}}}


{'usage_including_cached_inference': {'total_cost': 0.000152, 'gpt-3.5-turbo-0125': {'cost': 0.000152, 'prompt_tokens': 202, 'completion_tokens': 34, 'total_tokens': 236}}, 'usage_excluding_cached_inference': {'total_cost': 0.000152, 'gpt-3.5-turbo-0125': {'cost': 0.000152, 'prompt_tokens': 202, 'completion_tokens': 34, 'total_tokens': 236}}}


{'usage_including_cached_inference': {'total_cost': 0.0005819999999999999, 'gpt-3.5-turbo-0125': {'cost': 0.0005819999999999999, 'prompt_tokens': 558, 'completion_tokens': 202, 'total_tokens': 760}}, 'usage_excluding_cached_inference': {'total_cost': 0.00029099999999999997, 'gpt-3.5-turbo-0125': {'cost': 0.